# 自己検証する AI 要約パイプラインを試す

Databricks Technical Blog「Building a Self-Validating AI Summary Pipeline on Databricks」の構成を、小さなサンプルデータで動かします。

流れは次のとおりです。

1. A/B テストの結果テーブルから、統計的に有意な行だけを読む
2. MLflow Prompt Registry のプロンプトで要約を生成する
3. 4 段階の検証にかける (コード → 軽いモデル → 推論の強いモデル → 元データとの照合)
4. 失敗したら、失敗内容をプロンプトに足して作り直す (上限 3 回)
5. 一番良い試行を選び、PASS / SOFT FAIL / HARD FAIL を付ける
6. 一連の処理を MLflow トレースに残し、ジャッジ 2 種類の結果を同じトレースに書き込む

プロンプトは 2 種類登録します。ルールを書いていない雑なプロンプトと、ルールを書いたプロンプトです。雑なプロンプトでも検証と作り直しで要約が直る様子を見ます。

サーバーレスコンピュートで上から順に実行してください。

In [ ]:
%pip install -U -q "mlflow[databricks]>=3.4" openai

In [ ]:
dbutils.library.restartPython()

## 1. 設定

カタログとスキーマはウィジェットで変更できます。モデルは役割ごとに候補を並べてあり、応答した最初のものを使います。先頭がブログ記事の既定値です。

In [ ]:
import json
import re

import mlflow
import pandas as pd

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

RESULTS_TABLE = f"{CATALOG}.{SCHEMA}.svs_experiment_results"
SUMMARY_TABLE = f"{CATALOG}.{SCHEMA}.svs_summaries"
PROMPT_NAME = f"{CATALOG}.{SCHEMA}.svs_experiment_summary"
MAX_ATTEMPTS = 3

MODEL_CANDIDATES = {
    "generator": ["system.ai.claude-sonnet-5", "databricks-claude-sonnet-4-5", "databricks-meta-llama-3-3-70b-instruct"],
    "judge_simple": ["system.ai.claude-haiku-4-5", "databricks-claude-haiku-4-5", "databricks-meta-llama-3-3-70b-instruct"],
    "judge_complex": ["system.ai.claude-sonnet-4-5", "databricks-claude-sonnet-4-5", "databricks-meta-llama-3-3-70b-instruct"],
    "judge_data": ["system.ai.gemini-3-5-flash", "databricks-gemini-2-5-flash", "databricks-meta-llama-3-3-70b-instruct"],
}

# 実行結果をためておく。最後のまとめはこれを元に作る
RESULTS = []


def record(item, ok, message=""):
    print(f"[{'成功' if ok else '失敗'}] {item}")
    if message:
        print(f"       {message[:300]}")
    RESULTS.append({"item": item, "result": "成功" if ok else "失敗", "message": message[:200]})


def first_line(e):
    return str(e).strip().split("\n")[0]

## 2. モデルを呼ぶ関数と、使えるモデルの確認

In [ ]:
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()


def _text(content):
    """応答本文を文字列にそろえる (モデルによってはリストで返る)"""
    if isinstance(content, str):
        return content
    parts = []
    for p in content or []:
        if isinstance(p, dict) and p.get("type") == "text":
            parts.append(p.get("text", ""))
    return "".join(parts)


def _complete(model, prompt, max_tokens=1500):
    res = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
    )
    return _text(res.choices[0].message.content).strip()


MODELS = {}
for role, candidates in MODEL_CANDIDATES.items():
    for m in candidates:
        try:
            _complete(m, "「OK」とだけ返してください。", max_tokens=50)
            MODELS[role] = m
            record(f"モデル確認 {role}: {m}", True)
            break
        except Exception as e:
            print(f"  使えません: {m} / {first_line(e)[:200]}")
    if role not in MODELS:
        record(f"モデル確認 {role}", False, "候補のモデルがどれも応答しませんでした")

# 見つからなかった役割は、見つかったモデルで代用する
if MODELS:
    spare = list(MODELS.values())[0]
    for role in MODEL_CANDIDATES:
        if role not in MODELS:
            MODELS[role] = spare
            print(f"{role} は {spare} で代用します")

MODELS

## 3. サンプルの実験結果テーブル

指標とセグメントごとに、改善率 (lift) と有意かどうかを持つテーブルです。指標名はあえて略語 (CVR、AOV、ARPU) のまま入れています。

- EXP-2041: 改善と悪化の両方が有意
- EXP-2042: 一部のセグメントだけ有意
- EXP-2043: 複数の指標で有意
- EXP-2044: 有意な結果がない

In [ ]:
rows = [
    ("EXP-2041", "購入ボタンの色変更", "CVR", "全体", 3.20, 3.52, 10.0, 0.004, True),
    ("EXP-2041", "購入ボタンの色変更", "CVR", "新規ユーザー", 2.10, 2.52, 20.0, 0.001, True),
    ("EXP-2041", "購入ボタンの色変更", "CVR", "既存ユーザー", 4.50, 4.55, 1.1, 0.610, False),
    ("EXP-2041", "購入ボタンの色変更", "AOV", "全体", 5400.0, 5238.0, -3.0, 0.030, True),
    ("EXP-2042", "おすすめ商品の表示位置変更", "CVR", "全体", 3.10, 3.14, 1.3, 0.480, False),
    ("EXP-2042", "おすすめ商品の表示位置変更", "CVR", "モバイル", 2.80, 3.05, 8.9, 0.012, True),
    ("EXP-2042", "おすすめ商品の表示位置変更", "CVR", "デスクトップ", 3.60, 3.42, -5.0, 0.210, False),
    ("EXP-2042", "おすすめ商品の表示位置変更", "ARPU", "モバイル", 310.0, 329.0, 6.1, 0.041, True),
    ("EXP-2043", "送料無料ラインの引き下げ", "CVR", "全体", 3.30, 3.70, 12.1, 0.001, True),
    ("EXP-2043", "送料無料ラインの引き下げ", "AOV", "全体", 5600.0, 5150.0, -8.0, 0.002, True),
    ("EXP-2043", "送料無料ラインの引き下げ", "ARPU", "全体", 185.0, 190.5, 3.0, 0.045, True),
    ("EXP-2044", "フッターの文言変更", "CVR", "全体", 3.25, 3.27, 0.6, 0.820, False),
    ("EXP-2044", "フッターの文言変更", "AOV", "全体", 5500.0, 5480.0, -0.4, 0.760, False),
]
schema = (
    "experiment_id string, experiment_name string, metric string, segment string, "
    "control_value double, treatment_value double, lift_pct double, p_value double, is_significant boolean"
)

try:
    spark.sql(f"DROP TABLE IF EXISTS {RESULTS_TABLE}")
    spark.createDataFrame(rows, schema).write.saveAsTable(RESULTS_TABLE)
    record(f"実験結果テーブルの作成: {RESULTS_TABLE}", True)
    display(spark.table(RESULTS_TABLE))
except Exception as e:
    record(f"実験結果テーブルの作成: {RESULTS_TABLE}", False, first_line(e))

## 4. 有意な結果だけを読む

有意な行が 1 件もない実験はエラーにします。

In [ ]:
def load_experiment(experiment_id):
    pdf = (
        spark.table(RESULTS_TABLE)
        .filter(f"experiment_id = '{experiment_id}' AND is_significant")
        .orderBy("metric", "segment")
        .toPandas()
    )
    if pdf.empty:
        raise ValueError(f"{experiment_id}: 有意な結果がありません")
    lines = []
    for r in pdf.to_dict("records"):
        lines.append(
            f"- 指標: {r['metric']} / セグメント: {r['segment']} / 対照群: {r['control_value']} / "
            f"施策群: {r['treatment_value']} / 改善率: {r['lift_pct']:+.1f}% / p値: {r['p_value']}"
        )
    return {
        "experiment_id": experiment_id,
        "experiment_name": pdf["experiment_name"].iloc[0],
        "text": "\n".join(lines),
    }


print(load_experiment("EXP-2041")["text"])

try:
    load_experiment("EXP-2044")
except ValueError as e:
    print(f"\n想定どおりエラーになりました: {e}")

## 5. プロンプトを Prompt Registry に登録する

2 種類を同じ名前でバージョン違いとして登録します。ルールを書いた方に `production` のエイリアスを付けます。再実行するたびに新しいバージョンが増えます。

In [ ]:
TEMPLATE_ROUGH = """次の A/B テストの結果を要約してください。

実験ID: {{experiment_id}}
実験名: {{experiment_name}}

結果 (統計的に有意なもののみ):
{{data}}
"""

TEMPLATE_RULES = """あなたはデータアナリストです。次の A/B テストの結果を、事業部門の担当者向けに要約してください。

実験ID: {{experiment_id}}
実験名: {{experiment_name}}

結果 (統計的に有意なもののみ):
{{data}}

ルール:
- 400文字以内、です・ます調で書く
- 最初の文で、どの施策が何をどれだけ変えたかを述べる。続けてセグメント別や指標別の内訳を述べる
- 改善した結果だけでなく、悪化した結果も必ず書く
- 数値は上のデータのとおりに書く。データにない数値を書かない
- 一部のセグメントの結果を、全体に当てはまるように書かない
- 略語は使わず言い換える (CVR は購入転換率、AOV は平均注文額、ARPU はユーザーあたり売上)
- 実験ID (EXP-で始まる番号) は書かない
- 感嘆符と、「劇的」「圧倒的」のような大げさな表現は使わない
- 要約の本文だけを出力する
"""


class LocalPrompt:
    """Prompt Registry に登録できなかったときの代わり"""

    def __init__(self, template, version):
        self.template = template
        self.version = version

    def format(self, **kwargs):
        t = self.template
        for k, v in kwargs.items():
            t = t.replace("{{" + k + "}}", str(v))
        return t


def register(template, message, local_version):
    try:
        p = mlflow.genai.register_prompt(name=PROMPT_NAME, template=template, commit_message=message)
        record(f"プロンプト登録: {message} (バージョン {p.version})", True)
        return p
    except Exception as e:
        record(f"プロンプト登録: {message}", False, first_line(e))
        return LocalPrompt(template, local_version)


prompt_rough = register(TEMPLATE_ROUGH, "雑なプロンプト", "local-rough")
prompt_rules = register(TEMPLATE_RULES, "ルール入りプロンプト", "local-rules")

try:
    mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="production", version=prompt_rules.version)
    prompt_rules = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    record("production エイリアスの設定と読み込み", True)
except Exception as e:
    record("production エイリアスの設定と読み込み", False, first_line(e))

PROMPTS = {"雑なプロンプト": prompt_rough, "ルール入りプロンプト": prompt_rules}

## 6. 検証ルール

ルールは 4 段階です。`severity` が `hard` のルールに違反した要約は公開しません。`soft` は公開したうえで記録に残します。

| 段階 | 誰が判定するか | 見るもの |
|---|---|---|
| deterministic | Python のコード | 感嘆符、実験ID、略語、文字数 |
| simple | 軽いモデル | 文体、構成 |
| complex | 推論の強いモデル | 矛盾、重複、誤解を招く一般化 |
| data | 別のモデル | 元データとの数値の一致、悪化した結果の書き漏らし |

In [ ]:
RULES = {
    "no_exclamation": {"tier": "deterministic", "severity": "soft", "rule": "感嘆符 (! と ！) を使わない"},
    "no_internal_id": {"tier": "deterministic", "severity": "soft", "rule": "EXP-で始まる実験IDを書かない"},
    "no_raw_abbreviation": {"tier": "deterministic", "severity": "soft",
                            "rule": "CVR、AOV、ARPU という略語をそのまま書かない"},
    "max_length": {"tier": "deterministic", "severity": "soft", "rule": "400文字以内で書く"},
    "tone": {"tier": "simple", "severity": "soft",
             "rule": "です・ます調で書かれていて、「劇的」「圧倒的」のような大げさな表現がない"},
    "structure": {"tier": "simple", "severity": "soft",
                  "rule": "最初の文で、どの施策が何をどれだけ変えたかを述べている。そのあとに内訳を述べている"},
    "no_contradiction": {"tier": "complex", "severity": "hard", "rule": "要約の中に、互いに矛盾する記述がない"},
    "no_duplicate_findings": {"tier": "complex", "severity": "soft", "rule": "同じ結果を言い換えて繰り返していない"},
    "not_misleading": {"tier": "complex", "severity": "hard",
                       "rule": "一部のセグメントや指標の結果を、全体に当てはまるように書いていない"},
    "numbers_match": {"tier": "data", "severity": "hard",
                      "rule": "要約に書かれた数値がすべて元データと一致している。元データにない数値がない"},
    "no_cherry_picking": {"tier": "data", "severity": "hard",
                          "rule": "元データにある悪化した結果 (改善率がマイナスの行) を書き漏らしていない"},
}


def _failure(check, reason):
    return {"check": check, "tier": RULES[check]["tier"], "severity": RULES[check]["severity"], "reason": reason}


@mlflow.trace(span_type="EVALUATOR")
def check_deterministic(summary):
    failures = []
    if "!" in summary or "！" in summary:
        failures.append(_failure("no_exclamation", "感嘆符が含まれています"))
    ids = re.findall(r"EXP-\d+", summary)
    if ids:
        failures.append(_failure("no_internal_id", f"実験IDが含まれています: {', '.join(sorted(set(ids)))}"))
    abbr = re.findall(r"CVR|AOV|ARPU", summary)
    if abbr:
        failures.append(_failure("no_raw_abbreviation", f"略語が含まれています: {', '.join(sorted(set(abbr)))}"))
    if len(summary) > 400:
        failures.append(_failure("max_length", f"{len(summary)}文字あります"))
    return failures


def _llm_check(tier, model, summary, data_text=None):
    """指定した段階のルールをまとめてモデルに判定させる"""
    checks = {k: v for k, v in RULES.items() if v["tier"] == tier}
    rule_lines = "\n".join(f"- {k}: {v['rule']}" for k, v in checks.items())
    data_block = f"\n元データ:\n{data_text}\n" if data_text else ""
    prompt = f"""次の要約が、各ルールを満たしているかを判定してください。
{data_block}
要約:
{summary}

ルール:
{rule_lines}

JSON の配列だけを出力してください。各要素は {{"check": ルール名, "passed": true または false, "reason": 理由}} の形にします。
ルールごとに 1 要素、合計 {len(checks)} 要素です。"""
    raw = _complete(model, prompt)
    try:
        items = json.loads(raw[raw.find("["): raw.rfind("]") + 1])
    except Exception:
        return [{"check": f"{tier}_judge_unparseable", "tier": tier, "severity": "soft",
                 "reason": f"判定結果を読み取れませんでした: {raw[:100]}"}]
    failures = []
    for it in items:
        if isinstance(it, dict) and it.get("check") in checks and it.get("passed") is False:
            failures.append(_failure(it["check"], str(it.get("reason", ""))))
    return failures


@mlflow.trace(span_type="EVALUATOR")
def check_simple(summary):
    return _llm_check("simple", MODELS["judge_simple"], summary)


@mlflow.trace(span_type="EVALUATOR")
def check_complex(summary):
    return _llm_check("complex", MODELS["judge_complex"], summary)


@mlflow.trace(span_type="EVALUATOR")
def check_data_quality(summary, data_text):
    return _llm_check("data", MODELS["judge_data"], summary, data_text)


@mlflow.trace(span_type="CHAIN")
def validate_summary(summary, data_text):
    return (
        check_deterministic(summary)
        + check_simple(summary)
        + check_complex(summary)
        + check_data_quality(summary, data_text)
    )

## 7. 生成、検証、作り直しのループ

- 検証に失敗したら、失敗内容を文章にしてプロンプトの末尾に足し、作り直す
- 上限まで試したら、hard の失敗が最も少ない試行を選ぶ。同数なら失敗の総数が少ない方を選ぶ
- 選んだ試行に hard の失敗が残っていれば HARD FAIL とし、要約の代わりに定型文を返す

In [ ]:
FALLBACK_MESSAGE = "この実験の要約は自動検証を通過しなかったため、アナリストが確認中です。"


def convert_failures_to_feedback(failures):
    lines = [f"- {RULES[f['check']]['rule'] if f['check'] in RULES else f['check']} (指摘: {f['reason']})"
             for f in failures]
    return "前回の要約は次の点で検証に通りませんでした。すべて直して書き直してください。\n" + "\n".join(lines)


@mlflow.trace(span_type="LLM")
def generate_summary(prompt_text, feedback=None):
    if feedback:
        prompt_text = prompt_text + "\n\n" + feedback
    return _complete(MODELS["generator"], prompt_text)


def _count(failures):
    hard = sum(1 for f in failures if f["severity"] == "hard")
    return hard, len(failures)


@mlflow.trace(name="summarize_experiment", span_type="AGENT")
def summarize_experiment(experiment_id, prompt, max_attempts=MAX_ATTEMPTS):
    exp = load_experiment(experiment_id)
    base_prompt = prompt.format(
        experiment_id=exp["experiment_id"], experiment_name=exp["experiment_name"], data=exp["text"]
    )
    attempts = []
    feedback = None
    for n in range(1, max_attempts + 1):
        with mlflow.start_span(name=f"attempt_{n}", span_type="CHAIN") as span:
            span.set_inputs({"feedback": feedback})
            summary = generate_summary(base_prompt, feedback)
            failures = validate_summary(summary, exp["text"])
            span.set_outputs({"summary": summary, "failures": failures})
        attempts.append({"attempt": n, "summary": summary, "failures": failures})
        if not failures:
            break
        feedback = convert_failures_to_feedback(failures)

    best = min(attempts, key=lambda a: _count(a["failures"]))
    hard, total = _count(best["failures"])
    status = "HARD FAIL" if hard else ("SOFT FAIL" if total else "PASS")
    return {
        "experiment_id": experiment_id,
        "data_text": exp["text"],
        "summary": FALLBACK_MESSAGE if status == "HARD FAIL" else best["summary"],
        "status": status,
        "failures": best["failures"],
        "attempts": attempts,
        "attempt_count": len(attempts),
        "selected_attempt": best["attempt"],
    }

## 8. 実行する

3 つの実験を、2 種類のプロンプトでそれぞれ要約します。モデルを最大で 70 回ほど呼ぶので、数分かかります。

In [ ]:
EXPERIMENT_IDS = ["EXP-2041", "EXP-2042", "EXP-2043"]
RUNS = []

for label, prompt in PROMPTS.items():
    for exp_id in EXPERIMENT_IDS:
        try:
            out = summarize_experiment(exp_id, prompt)
            out["prompt"] = label
            out["prompt_version"] = str(prompt.version)
            out["trace_id"] = mlflow.get_last_active_trace_id()
            RUNS.append(out)
            record(f"要約 {exp_id} / {label}: {out['status']} (試行 {out['attempt_count']} 回)", True)
        except Exception as e:
            record(f"要約 {exp_id} / {label}", False, first_line(e))

## 9. 結果を見る

まず、実験とプロンプトごとの最終結果です。

In [ ]:
if RUNS:
    display(pd.DataFrame([
        {
            "experiment_id": r["experiment_id"],
            "prompt": r["prompt"],
            "status": r["status"],
            "attempt_count": r["attempt_count"],
            "selected_attempt": r["selected_attempt"],
            "remaining_failures": ", ".join(f["check"] for f in r["failures"]),
            "summary": r["summary"],
        }
        for r in RUNS
    ]))

次に、試行ごとの履歴です。何回目の試行でどのルールに違反し、作り直しで直ったかを確認できます。

In [ ]:
if RUNS:
    display(pd.DataFrame([
        {
            "experiment_id": r["experiment_id"],
            "prompt": r["prompt"],
            "attempt": a["attempt"],
            "failed_checks": ", ".join(f"{f['check']} ({f['severity']})" for f in a["failures"]) or "なし",
            "reasons": " / ".join(f["reason"] for f in a["failures"]),
            "summary": a["summary"],
        }
        for r in RUNS
        for a in r["attempts"]
    ]))

エクスペリメントのトレース画面を開くと、`summarize_experiment` の下に `attempt_1`、`attempt_2` が並び、それぞれの中に生成と各段階の検証が入っています。

最終結果を Delta テーブルに保存します。

In [ ]:
if RUNS:
    try:
        out_pdf = pd.DataFrame([
            {
                "experiment_id": r["experiment_id"],
                "prompt": r["prompt"],
                "prompt_version": r["prompt_version"],
                "status": r["status"],
                "attempt_count": r["attempt_count"],
                "selected_attempt": r["selected_attempt"],
                "summary": r["summary"],
                "failures_json": json.dumps(r["failures"], ensure_ascii=False),
                "trace_id": r["trace_id"] or "",
            }
            for r in RUNS
        ])
        spark.sql(f"DROP TABLE IF EXISTS {SUMMARY_TABLE}")
        spark.createDataFrame(out_pdf).write.saveAsTable(SUMMARY_TABLE)
        record(f"要約テーブルの保存: {SUMMARY_TABLE}", True)
    except Exception as e:
        record(f"要約テーブルの保存: {SUMMARY_TABLE}", False, first_line(e))

## 10. ジャッジ 2 種類の結果をトレースに書き込む

- `usefulness`: 最終的な要約の有用性を 1 から 3 で採点する
- `check_diagnosis`: 1 回目の試行で失敗したチェックを見直し、検証側の誤判定だったものを挙げる。検証ルールの文言を直す材料になる

どちらも `make_judge` で作り、結果は要約を作ったときのトレースに書き込みます。

In [ ]:
from mlflow.entities import AssessmentSource
from mlflow.genai.judges import make_judge

JUDGE_MODEL = "databricks:/" + MODELS["judge_complex"]

USEFULNESS_INSTRUCTIONS = """A/B テスト結果の要約が、事業部門の担当者にとってどれだけ有用かを 1 から 3 の整数で採点してください。
3: 結論、内訳、悪化した結果がそろっていて、そのまま意思決定に使える
2: おおむね正しいが、読み手が元データを見直す必要がある
1: 誤りや抜けがあり、意思決定に使えない

実験の元データ: {{ inputs }}
要約: {{ outputs }}"""

DIAGNOSIS_INSTRUCTIONS = """要約の自動検証で失敗と判定されたチェックを見直してください。
inputs には、実験の元データと、失敗したチェックの一覧 (チェック名、ルールの文、指摘) が入っています。outputs は検証された要約です。

ルールの文だけを基準にして、誤判定されたチェックを探してください。誤判定とは、要約がルールを実際には満たしている場合、または指摘がルールの範囲外のことを問題にしている場合です。
誤判定されたチェック名をカンマ区切りで答えてください。誤判定がなければ none と答えてください。
理由には、要約の該当する語句と、ルールの文をどう変えれば誤判定を防げるかを書いてください。

inputs: {{ inputs }}
outputs: {{ outputs }}"""


def build_judge(name, instructions, value_type):
    try:
        return make_judge(name=name, instructions=instructions, model=JUDGE_MODEL, feedback_value_type=value_type)
    except TypeError:
        return make_judge(name=name, instructions=instructions, model=JUDGE_MODEL)


def log_to_trace(trace_id, name, fb):
    mlflow.log_feedback(
        trace_id=trace_id,
        name=name,
        value=fb.value,
        rationale=fb.rationale,
        source=AssessmentSource(source_type="LLM_JUDGE", source_id=JUDGE_MODEL),
    )


JUDGE_ROWS = []
try:
    usefulness_judge = build_judge("usefulness", USEFULNESS_INSTRUCTIONS, int)
    diagnosis_judge = build_judge("check_diagnosis", DIAGNOSIS_INSTRUCTIONS, str)
    record("ジャッジの作成", True)
except Exception as e:
    usefulness_judge = diagnosis_judge = None
    record("ジャッジの作成", False, first_line(e))

if usefulness_judge:
    for r in RUNS:
        key = f"{r['experiment_id']} / {r['prompt']}"
        row = {"experiment_id": r["experiment_id"], "prompt": r["prompt"], "status": r["status"]}
        try:
            fb = usefulness_judge(inputs={"experiment_data": r["data_text"]}, outputs=r["summary"])
            row["usefulness"] = str(fb.value)
            row["usefulness_rationale"] = fb.rationale
            if r["trace_id"]:
                log_to_trace(r["trace_id"], "usefulness", fb)
            record(f"usefulness {key}", True)
        except Exception as e:
            record(f"usefulness {key}", False, first_line(e))

        first = r["attempts"][0]
        if first["failures"]:
            try:
                failed = [
                    {"check": f["check"], "rule": RULES.get(f["check"], {}).get("rule", ""), "reason": f["reason"]}
                    for f in first["failures"]
                ]
                fb = diagnosis_judge(
                    inputs={"experiment_data": r["data_text"], "failed_checks": failed},
                    outputs=first["summary"],
                )
                row["first_attempt_failed_checks"] = ", ".join(f["check"] for f in first["failures"])
                row["check_diagnosis"] = str(fb.value)
                row["check_diagnosis_rationale"] = fb.rationale
                if r["trace_id"]:
                    log_to_trace(r["trace_id"], "check_diagnosis", fb)
                record(f"check_diagnosis {key}", True)
            except Exception as e:
                record(f"check_diagnosis {key}", False, first_line(e))
        JUDGE_ROWS.append(row)

if JUDGE_ROWS:
    display(pd.DataFrame(JUDGE_ROWS).fillna(""))

## 11. 実行結果のまとめ

In [ ]:
if RESULTS:
    summary_df = pd.DataFrame(RESULTS)
    ok = (summary_df["result"] == "成功").sum()
    ng = (summary_df["result"] == "失敗").sum()
    print(f"実行した項目: {len(summary_df)} 件 (成功 {ok} / 失敗 {ng})")
    display(summary_df)
    if ng:
        display(summary_df[summary_df["result"] == "失敗"][["item", "message"]])

## 後片付け

テーブルを消すときは、コメントを外して実行してください。プロンプトはカタログエクスプローラーから削除できます。

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {RESULTS_TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {SUMMARY_TABLE}")